In [227]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [228]:
df=pd.read_csv('./data/cleaned_credit_data.csv')
df.head()

,Customer_ID,Month,Age,Occupation,Annual_Income,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Delay_from_due_date,Num_of_Delayed_Payment,...,Investment_Ratio,Spending_Level,Payment_Value_Level,Num_Loan_Types,Has_Mortgage,Has_Student,Has_Personal,Has_Auto,Has_Debt_Consolidation,Has_Credit_Builder
0,CUS_0xd40,1,23.0,Scientist,19114.12,3,4,3,3,7.0,...,0.050485,High,Small,4.0,0,0,1,1,0,1
1,CUS_0xd40,3,NaN,Scientist,19114.12,3,4,3,3,7.0,...,0.051292,Low,Medium,4.0,0,0,1,1,0,1
2,CUS_0xd40,4,23.0,Scientist,19114.12,3,4,3,5,4.0,...,0.125221,Low,Small,4.0,0,0,1,1,0,1
3,CUS_0xd40,6,23.0,Scientist,19114.12,3,4,3,8,4.0,...,0.039194,NaN,NaN,4.0,0,0,1,1,0,1
4,CUS_0xd40,7,23.0,Scientist,19114.12,3,4,3,3,8.0,...,0.111966,Low,Small,4.0,0,0,1,1,0,1


In [229]:
df.isnull().mean()*100

Customer_ID                    0.000000
Month                          0.000000
Age                            8.588476
Occupation                     7.023990
Annual_Income                  0.000000
Num_Bank_Accounts              0.000000
Num_Credit_Card                0.000000
Interest_Rate                  0.000000
Delay_from_due_date            0.000000
Num_of_Delayed_Payment         0.000000
Changed_Credit_Limit           2.097606
Num_Credit_Inquiries           1.973045
Credit_Mix                    20.323360
Outstanding_Debt               0.000000
Credit_Utilization_Ratio       0.000000
Payment_of_Min_Amount         11.929200
Total_EMI_per_month            0.000000
Monthly_Balance                2.808849
Credit_Score                   0.000000
Credit_History_Age(months)     9.039386
Investment_Ratio               0.000000
Spending_Level                 7.532199
Payment_Value_Level            7.532199
Num_Loan_Types                11.378640
Has_Mortgage                   0.000000


In [230]:
df["Debt_to_Income"] = df["Outstanding_Debt"] / df["Annual_Income"]

df["EMI_to_Income"] = (df["Total_EMI_per_month"] * 12) / df["Annual_Income"]

df["Delay_Intensity"] = df["Num_of_Delayed_Payment"] / (df["Num_Loan_Types"] + 1)

df["Utilization_Delay"] = df["Credit_Utilization_Ratio"] * df["Num_of_Delayed_Payment"]

df["Debt_per_Loan"] = df["Outstanding_Debt"] / (df["Num_Loan_Types"] + 1)

In [231]:
df.drop(columns=['Outstanding_Debt','Annual_Income','Total_EMI_per_month'],inplace=True)

In [232]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Credit_Score"] = le.fit_transform(df["Credit_Score"])

In [233]:
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,classification_report,ConfusionMatrixDisplay,f1_score

In [234]:
unique_ids = df["Customer_ID"].unique()

train_ids, test_ids = train_test_split(
    unique_ids,
    test_size=0.2,
    random_state=42
)

In [235]:
train_df = df[df["Customer_ID"].isin(train_ids)].copy()
test_df  = df[df["Customer_ID"].isin(test_ids)].copy()

train_df.drop(columns=["Customer_ID"], inplace=True)
test_df.drop(columns=["Customer_ID"], inplace=True)

x_train = train_df.drop("Credit_Score", axis=1)
y_train = train_df["Credit_Score"]

x_test = test_df.drop("Credit_Score", axis=1)
y_test = test_df["Credit_Score"]

In [236]:
x_train.head()

,Month,Age,Occupation,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,...,Has_Student,Has_Personal,Has_Auto,Has_Debt_Consolidation,Has_Credit_Builder,Debt_to_Income,EMI_to_Income,Delay_Intensity,Utilization_Delay,Debt_per_Loan
6,1,28.0,NaN,2,4,6,3,4.0,5.42,2.0,...,0,0,0,0,1,0.017362,0.006479,2.0,97.856123,302.515
7,2,28.0,Teacher,2,4,6,7,1.0,7.42,2.0,...,0,0,0,0,1,0.017362,0.006479,0.5,38.550848,302.515
8,4,28.0,Teacher,2,4,6,3,3.0,5.42,2.0,...,0,0,0,0,1,0.017362,0.006479,1.5,117.547967,302.515
9,5,28.0,Teacher,2,4,6,3,1.0,6.42,2.0,...,0,0,0,0,1,0.017362,0.006479,0.5,34.977895,302.515
10,6,28.0,Teacher,2,4,6,3,0.0,5.42,2.0,...,0,0,0,0,1,0.017362,0.006479,0.0,0.000000,302.515


In [237]:
num_cols=x_train.select_dtypes(include='number').columns
cat_cols=x_train.select_dtypes(exclude='number').columns

In [238]:
for col in cat_cols:
    print(col,":",x_train[col].unique())

Occupation : [nan 'Teacher' 'Engineer' 'Developer' 'Lawyer' 'Media_Manager'
 'Journalist' 'Entrepreneur' 'Manager' 'Scientist' 'Accountant' 'Musician'
 'Mechanic' 'Writer' 'Architect' 'Doctor']
Credit_Mix : ['Good' nan 'Standard' 'Bad']
Payment_of_Min_Amount : ['No' nan 'Yes']
Spending_Level : ['Low' 'High' nan]
Payment_Value_Level : ['Small' 'Large' 'Medium' nan]


In [239]:
cat_cols

Index(['Occupation', 'Credit_Mix', 'Payment_of_Min_Amount', 'Spending_Level',
       'Payment_Value_Level'],
      dtype='object')

In [240]:
ohe_cols=['Occupation']
oe_cols=['Credit_Mix', 'Payment_of_Min_Amount', 'Spending_Level','Payment_Value_Level']

In [241]:
categories=[
    ['Bad','Standard','Good'],
    ['No','Yes'],
    ['Low','High'],
    ['Small','Medium','Large']
]

num_trf=(
    'num_transform',Pipeline([
    ('impute',SimpleImputer(strategy='median')),
    ('scale',StandardScaler()),
]),num_cols
)
oe_trf=(
    'oe_transform',Pipeline([
    ('impute',SimpleImputer(strategy='most_frequent')),
    ('encode',OrdinalEncoder(categories=categories)),
]),oe_cols
)
ohe_trf=(
    'col_transform',Pipeline([
    ('impute',SimpleImputer(strategy='most_frequent')),
    ('encode',OneHotEncoder(handle_unknown='ignore')),
]),ohe_cols
)

In [242]:
transformer=ColumnTransformer(transformers=[num_trf,oe_trf,ohe_trf],remainder='passthrough')

In [243]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import AdaBoostClassifier

In [244]:
models = {
    "Logistic Regression": LogisticRegression(
        random_state=42,
        max_iter=1000,
        multi_class='multinomial'
    ),
    "Random Forest": RandomForestClassifier(
        class_weight='balanced',
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        random_state=42,
        objective='multi:softprob',
        num_class=3,
        eval_metric='mlogloss'
    ),
    "LightGBM": LGBMClassifier(
        random_state=42,
        class_weight='balanced'
    ),
    "CatBoost": CatBoostClassifier(
        random_state=42,
        verbose=0
    ),
    "SGD Classifier": SGDClassifier(
        loss='log_loss',
        random_state=42,
        max_iter=1000
    ),
    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),
    "AdaBoost": AdaBoostClassifier(
        random_state=42
    )
}

In [245]:
model_list=[]
macro_f1=[]
accuracy=[]

for i in range(len(list(models))):
    clf = list(models.values())[i]
    model=Pipeline([
        ('transformer',transformer),
        ("clf",clf)
    ])
    model.fit(x_train, y_train)

    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    train_macro_f1=f1_score(y_train, y_train_pred, average='macro')
    test_macro_f1=f1_score(y_test, y_test_pred, average='macro')

    train_accuracy=accuracy_score(y_train,y_train_pred)
    test_accuracy=accuracy_score(y_test,y_test_pred)

    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])

    print('Model performance for Training set')
    print("- Macro F1: {:.4f}".format(train_macro_f1))
    print("- Accuracy SCore: {:.4f}".format(train_accuracy))

    print("___________________________________________________")

    print('Model performance for Test set')
    print("- Macro F1: {:.4f}".format(test_macro_f1))
    print("- Accuracy SCore: {:.4f}".format(test_accuracy))

    print("="*35)

    macro_f1.append(test_macro_f1)
    accuracy.append(test_accuracy)

Logistic Regression
Model performance for Training set
- Macro F1: 0.5770
- Accuracy SCore: 0.6230
___________________________________________________
Model performance for Test set
- Macro F1: 0.5563
- Accuracy SCore: 0.5980
Random Forest
Model performance for Training set
- Macro F1: 1.0000
- Accuracy SCore: 1.0000
___________________________________________________
Model performance for Test set
- Macro F1: 0.6536
- Accuracy SCore: 0.6808
XGBoost
Model performance for Training set
- Macro F1: 0.8307
- Accuracy SCore: 0.8398
___________________________________________________
Model performance for Test set
- Macro F1: 0.6505
- Accuracy SCore: 0.6773
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003450 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3209
[LightGBM] [Info] Number of data points in the train set: 64249, number of used features: 43
[LightGBM] [Info] Start training from score -1.0986

In [248]:
score=pd.DataFrame(list(zip(model_list, accuracy,macro_f1)), columns=['Model Name', 'Accuracy',"Macro F1"])

In [247]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, accuracy_score
from sklearn.pipeline import Pipeline

model_list = []
grid_macro_f1 = []
grid_accuracy = []

param_grids = {
    "Logistic Regression": {
        "clf__C": [0.1, 1, 10]
    },
    "Random Forest": {
        "clf__n_estimators": [200, 400],
        "clf__max_depth": [None, 20, 30]
    },
    "XGBoost": {
        "clf__n_estimators": [300, 600],
        "clf__max_depth": [4, 6],
        "clf__learning_rate": [0.05, 0.1]
    },
    "LightGBM": {
        "clf__n_estimators": [300, 600],
        "clf__max_depth": [-1, 20],
        "clf__learning_rate": [0.05, 0.1]
    },
    "CatBoost": {
        "clf__depth": [4, 6],
        "clf__learning_rate": [0.05, 0.1]
    },
    "SGD Classifier": {
        "clf__alpha": [0.0001, 0.001]
    },
    "KNN": {
        "clf__n_neighbors": [3, 5, 7]
    },
    "AdaBoost": {
        "clf__n_estimators": [100, 300]
    }
}

for name, clf in models.items():

    pipeline = Pipeline([
        ('transformer', transformer),
        ('clf', clf)
    ])

    grid = GridSearchCV(
        pipeline,
        param_grid=param_grids[name],
        cv=3,
        scoring='f1_macro',
        n_jobs=-1
    )

    grid.fit(x_train, y_train)

    best_model = grid.best_estimator_

    y_train_pred = best_model.predict(x_train)
    y_test_pred = best_model.predict(x_test)

    train_macro_f1 = f1_score(y_train, y_train_pred, average='macro')
    test_macro_f1 = f1_score(y_test, y_test_pred, average='macro')

    train_accuracy = accuracy_score(y_train, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)

    print(name)
    print("Best Params:", grid.best_params_)
    print("Train Macro F1:", round(train_macro_f1, 4))
    print("Test Macro F1:", round(test_macro_f1, 4))
    print("Train Accuracy:", round(train_accuracy, 4))
    print("Test Accuracy:", round(test_accuracy, 4))
    print("="*40)

    model_list.append(name)
    grid_macro_f1.append(test_macro_f1)
    grid_accuracy.append(test_accuracy)

Logistic Regression
Best Params: {'clf__C': 10}
Train Macro F1: 0.577
Test Macro F1: 0.5563
Train Accuracy: 0.623
Test Accuracy: 0.5979
Random Forest
Best Params: {'clf__max_depth': 20, 'clf__n_estimators': 400}
Train Macro F1: 0.9375
Test Macro F1: 0.6629
Train Accuracy: 0.9398
Test Accuracy: 0.6742
XGBoost
Best Params: {'clf__learning_rate': 0.05, 'clf__max_depth': 4, 'clf__n_estimators': 300}
Train Macro F1: 0.7009
Test Macro F1: 0.6609
Train Accuracy: 0.7249
Test Accuracy: 0.6828
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003464 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3209
[LightGBM] [Info] Number of data points in the train set: 64249, number of used features: 43
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
LightGBM
Best Params: {'clf__learning_rate': 0.05, 'clf__max

In [251]:
score['Grid_f1']=grid_macro_f1
score['Grid_accuracy']=grid_accuracy
score.sort_values(by='Grid_f1',ascending=False)

,Model Name,Accuracy,Macro F1,Grid_f1,Grid_accuracy
3,LightGBM,0.665939,0.662614,0.664434,0.668122
1,Random Forest,0.680783,0.653597,0.662907,0.674234
4,CatBoost,0.681719,0.657532,0.662305,0.683465
2,XGBoost,0.677291,0.650510,0.660906,0.682842
7,AdaBoost,0.610865,0.576814,0.571637,0.609555
0,Logistic Regression,0.597954,0.556326,0.556315,0.597892
5,SGD Classifier,0.592466,0.551232,0.554248,0.596956
6,KNN,0.570448,0.532347,0.543094,0.585106
